# Image scoring from a brand JSON file

Set `IMAGE_PATH` and `CONFIG_PATH`, then **Run All**. The output is the overall score, individual criterion scores, and optional CV measurements.

The notebook calls the existing image loader, preprocessor, config bridge, CV criteria, and scoring engine. The default JSON contains brightness and contrast only. To use the original full rubric, select `config/brand_standards.json` below. Full-rubric rows needing subject/model inputs or unset thresholds remain unavailable, with a partial overall status. Thresholds are the existing draft values.


In [1]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import HTML, display

ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src" / "brandlens").is_dir()
    and (path / "config" / "brand_standards.json").is_file()
)

In [2]:
from brandlens.config import load_cv_config
from brandlens.cv.cv_input import CVInput
from brandlens.preprocessing.image_io import load_image
from brandlens.preprocessing.preprocessor import preprocess_image
from brandlens.scoring import score_image

In [3]:
# Change these two paths to use your image and selected brand configuration.
IMAGE_PATH = ROOT / "examples" / "data" / "backend_scoring_sample.png"
CONFIG_PATH = ROOT / "config" / "brand_standards_brightness_contrast_test.json"
# CONFIG_PATH = ROOT / "config" / "brand_standards.json"

SCHEMA_PATH = ROOT / "config" / "brand_standards.schema.json"
SHOW_MEASUREMENTS = True

configuration = load_cv_config(
    CONFIG_PATH,
    schema_path=SCHEMA_PATH,
)

In [4]:
def score_photo(image, configuration):
    """Preprocess and score an already-loaded Pillow image.

    Args:
        image: Caller-owned Pillow image, before preprocessing.
        configuration: The (rubric, criteria) tuple returned by
            load_cv_config().

    Returns:
        tuple: The backend ImageScore and CV result dictionaries.

    Raises:
        ValueError: If preprocessing excludes the image by geometry.
        Other preprocessing, analysis, and scoring errors propagate.
    """
    rubric, criteria = configuration

    # Preprocessing returns a separate standardized image.
    standardized = preprocess_image(image)

    if standardized is None:
        raise ValueError(
            "Image excluded by the preprocessor's geometry filter."
        )

    # Close the image created here. The caller owns the original image.
    with standardized:
        data = CVInput(standardized)

        cv_results = {
            cid: criterion.analyze(data)
            for cid, criterion in criteria.items()
        }

    scored = score_image(rubric, cv_results=cv_results)

    return scored, cv_results

In [5]:
with load_image(IMAGE_PATH) as image:
    scored, cv_results = score_photo(image, configuration)

# Overall score and coverage. Partial results stay clearly identified.
overall = pd.DataFrame(
    [
        {
            "rubric_id": scored.rubric_id,
            "overall_score": scored.overall_score,
            "status": scored.status,
            "scored_criteria": sum(row.availability == "ok" for row in scored.criteria),
            "configured_criteria": len(scored.criteria),
        }
    ]
)
display(
    HTML(
        "<h4>Overall score (0–100)</h4>"
        + overall.to_html(index=False, na_rep="—", float_format="{:.2f}".format)
    )
)

individual = pd.DataFrame(
    [
        {
            "criterion_id": row.criterion_id,
            "score": row.score,
            "applied_weight": row.weight,
            "availability": row.availability,
            "reason": "; ".join(row.reasons),
        }
        for row in scored.criteria
    ]
)
display(
    HTML(
        "<h4>Individual scores (0–100)</h4>"
        + individual.to_html(index=False, na_rep="—", float_format="{:.2f}".format)
    )
)

if SHOW_MEASUREMENTS:
    measurement_rows = [
        {
            "criterion_id": cid,
            "measurement": name,
            "value": json.dumps(value) if isinstance(value, (list, dict)) else value,
            "unit": result["units"].get(name, ""),
        }
        for cid, result in cv_results.items()
        for name, value in result["measurements"].items()
    ]
    measurements = pd.DataFrame(
        measurement_rows, columns=["criterion_id", "measurement", "value", "unit"]
    )
    display(
        HTML(
            "<h4>CV measurements</h4>"
            + measurements.to_html(
                index=False, na_rep="—", float_format="{:.4f}".format
            )
        )
    )

rubric_id,overall_score,status,scored_criteria,configured_criteria
brightness-contrast-demo,100.00,scored,2,2


criterion_id,score,applied_weight,availability,reason
exposure_brightness,100.00,0.50,ok,
contrast,100.00,0.50,ok,


criterion_id,measurement,value,unit
exposure_brightness,mean_luminance,99.0000,grayscale_0_255
exposure_brightness,near_black_fraction,0.0000,fraction_0_1
exposure_brightness,near_white_fraction,0.0000,fraction_0_1
contrast,luminance_std,58.1227,grayscale_0_255
contrast,p5_p95_range,163.0000,grayscale_0_255
contrast,near_black_fraction,0.0000,fraction_0_1
contrast,near_white_fraction,0.0000,fraction_0_1
